# Лабораторная работа 1

In [ ]:
import pandas as pd
import numpy as np
from numpy.typing import NDArray
from typing import Sequence, List

data = pd.read_csv("/content/olist_order_items_dataset.csv")

prices = data['price'].values
freight_value = data['freight_value'].values

**Задание 1: Битва производительности (Loops vs NumPy)**

In [ ]:
def discount(price: float) -> float:
  if price > 500: return 0.15
  if price > 100: return 0.1
  return 0.0


#Реализация А
def task1_A(
    prices: Sequence[float],
    freight_value: Sequence[float]
) -> List[float]:

  result: List[float] = []
  for i in range(len(prices)):
    price = prices[i]
    f = freight_value[i]

    total = (price * (1 - discount(price))) + f
    result.append(total)


#Реализация B
def task1_B(
    prices: NDArray[np.number],
    freight_value: NDArray[np.number]
) -> NDArray[np.number]:
  cond = [
      prices > 500,
      prices > 100
  ]
  discounts = [0.15, 0.1]
  discount_arr = np.select(cond, discounts, default=0.0)

  return (prices * (1 - discount_arr) + freight_value)

In [ ]:
#Профилирование
t_python = %timeit -o task1_A(prices, freight_value)
t_numpy = %timeit -o task1_B(prices, freight_value)

res = t_python.average / t_numpy.average
print("Сравнение скорости: ", res)

68.5 ms ± 1.11 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
1.52 ms ± 169 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)
Сравнение скорости:  44.99781138311909


**Задание 2: Интеллектуальный фильтр и Fancy Indexing**

In [ ]:
# Задание 2
mean_price = prices.mean() #среднее
std_price = prices.std() #стандартное отклонение

anomaly_threshold = mean_price + 3 * std_price
result = data.values[prices > anomaly_threshold]

random_ind = np.random.randint(len(prices), size = 100)
random_orders = data.values[random_ind]
random_orders

array([['22fe58e8a8084630f52ce3fbb02c9ac1', 1,
        '6dde44b4172999f35f08654d06bad633',
        '7c67e1448b00f6e969d365cea6b010ab', '2017-11-21 12:50:39',
        194.99, 35.53],
       ['a865adb8bec2475e30e835eaf1b9e7d6', 1,
        '16078eb07519685083510f372565ddc7',
        'eb4df17aed01d918c65f0f8d650900c0', '2018-05-11 10:53:01',
        318.98, 20.11],
       ['887b5b39f6f5c08315aa901f0fadba6d', 1,
        '45804253761d5c941dda4383a2b35e8d',
        '89c127985a8b130cfa45c1d36764017a', '2018-01-05 12:08:16', 94.9,
        21.06],
       ['ee9c09afa667c1baabeefdf51f66148e', 1,
        'fc9f1b7e877d87c6e78cc32686a73a27',
        'cce6ab8d1682639fe45ab70234f1665f', '2017-07-05 11:15:14', 34.9,
        18.59],
       ['79d51aa9809de1c33dc6cfdf113de4bd', 1,
        'b2545dd62c61d45d55ebaa04d1cb1e23',
        'e48b04bf1aa1ef711caebeb7aede6180', '2017-11-30 22:52:58', 98.9,
        17.94],
       ['4310dee37e09d3d73cf8c9a63f7b61fc', 1,
        '98c2f7da94217786e372e7d85462c354',
     

**Задание 3: Бродкастинг (Объемный вес и логистика)**

In [ ]:
data2 = pd.read_csv("/content/olist_products_dataset.csv")

L = data2['product_length_cm'].values
H = data2['product_height_cm'].values
W = data2['product_width_cm'].values
Weight = data2['product_weight_g'].values

In [ ]:
Vweight = (L*H*W) / 6000
Vweight

array([0.37333333, 1.8       , 0.405     , ..., 0.8505    , 1.34333333,
       0.028     ])

In [ ]:
#Представим, что матрицы такие:
price_arr = np.random.uniform(10, 1000, (1000, 5))
region_coef = [1.0, 1.1, 0.95, 1.2, 0.9]
adjusted_prices = price_arr * region_coef
adjusted_prices

array([[ 759.73856321,   63.8290963 ,  211.42861013,  179.42132249,
         767.14211251],
       [ 320.03203457,  200.43753526,  741.18708831,  144.3942949 ,
         452.70594437],
       [ 149.50454918,  469.40756475,  453.23896779,  990.30029853,
         629.41830497],
       ...,
       [ 731.04050804,  924.59467431,  547.31469783, 1184.46725492,
         171.50343074],
       [ 781.21126588,  920.73252674,  632.34890071,  574.96070708,
         770.03587308],
       [ 266.77553918,  744.96890715,  456.63367856,  710.19037258,
          62.438821  ]])

**Задание 4: Векторное сходство (Линейная алгебра)**

In [ ]:
matrix = np.vstack((Weight[:1000], L[:1000], H[:1000], W[:1000]))
mean = np.mean(matrix)
std_dev = np.std(matrix)
normal_matrix = (matrix - mean) / std_dev

target_vector = normal_matrix[0]
A = np.linalg.norm(normal_matrix)
B = np.linalg.norm(target_vector)
AxB = np.dot(normal_matrix, target_vector)

cosine_similarity =  AxB / (A * B)
cosine_similarity

array([ 0.97785944, -0.04254138, -0.04378667, -0.0433501 ])